In [1]:
# SECCION 4. PRIMER PROGRAMA "HOLA SPARK"

In [17]:
from pyspark.sql import SparkSession

# 1. Crear SparkSession
spark = SparkSession.builder.appName("HolaSpark") \
    .config("spark.ui.port", "4040") \
    .config("spark.ui.allowFraming", "true") \
    .getOrCreate()

# 2. Obtener el SparkContext (para funciones de bajo nivel)
sc = spark.sparkContext

# 3. Crear un RDD (Resilient Distributed Dataset)
# parallelize() distribuye los datos en particiones
numeros = sc.parallelize(range(101))

# 4. Realizar una acción: sumar todos los números
total = numeros.sum()
print(f"La suma de 0 a 100 es: {total}")
# Salida: La suma de 0 a 100 es: 5050

# 5. Verificar cuántas particiones se crearon
print(f"Número de particiones: {numeros.getNumPartitions()}")

La suma de 0 a 100 es: 5050
Número de particiones: 12


In [19]:
# SECCION 5. Spark UI y Jobs y Stages
rdd_grande = sc.parallelize(range(1000000), 10)
resultado = rdd_grande.filter(lambda x: x % 2 == 0).count()

In [18]:
spark

In [16]:
spark.stop()

In [20]:
# SECCION 6. LECTURA DE CSV

In [21]:
# Crear carpeta y descargar datos
!mkdir -p DATOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/pdi_sales_small.csv" -P ./DATOS

# Ver las primeras líneas
!head ./DATOS/pdi_sales_small.csv

--2026-11-19 00:02:12--  https://github.com/josepgarcia/datos/raw/refs/heads/main/pdi_sales_small.csv
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/pdi_sales_small.csv [following]
--2026-11-19 00:02:12--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/pdi_sales_small.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5549121 (5.3M) [text/plain]
Saving to: ‘./DATOS/pdi_sales_small.csv’

pdi_sales_small.csv 100%[===================>]   5.29M  8.83MB/s    in 0.6s    

2026-11-19 00:02:14 (8.83 MB/s) - ‘./DATOS/pdi_sales_small.csv’ saved [554

In [24]:
# Cargar y Procesar

In [25]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AnalisisVentas").getOrCreate()
sc = spark.sparkContext

# Construir ruta absoluta
path_relativo = "./DATOS/pdi_sales_small.csv"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

print(f"Leyendo desde: {path_final}")

# Leer archivo (cada línea es un string)
rdd = sc.textFile(path_final)

# Ver las primeras 3 líneas
print(rdd.take(3))
# Salida:
# ['ProductID;Date;Zip;Units;Revenue;Country',
#  '725;1/15/1999;41540          ;1;115.5;Germany',
#  '787;6/6/2002;41540          ;1;314.9;Germany']

Leyendo desde: file:///home/jovyan/work/projects/DATOS/pdi_sales_small.csv
['ProductID;Date;Zip;Units;Revenue;Country', '725;1/15/1999;41540          ;1;115.5;Germany', '787;6/6/2002;41540          ;1;314.9;Germany']


In [26]:
# Paso 3
# Extraer (País, Unidades) de cada línea
# Formato: ProductID;Date;Zip;Units;Revenue;Country
#          0         1    2   3     4       5

paises_unidades = rdd.map(lambda linea: (
    linea.split(";")[-1].strip(),  # País (última columna)
    linea.split(";")[3]             # Unidades (columna 3)
))

paises_unidades.take(5)
# Salida:
# [('Country', 'Units'),
#  ('Germany', '1'),
#  ('Germany', '1'),
#  ...]

[('Country', 'Units'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1')]

In [27]:
# Paso 4 Optimizacion
# ✅ MEJOR: Función que hace split una sola vez
def parsear_linea(linea):
    columnas = linea.split(";")
    pais = columnas[-1].strip()
    unidades = columnas[3]
    return (pais, unidades)

paises_unidades = rdd.map(parsear_linea)
paises_unidades.take(5)

[('Country', 'Units'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1')]

In [28]:
# Paso 5 Eliminar Encabezado
# Obtener la primera línea (encabezado)
header = paises_unidades.first()
print(f"Encabezado: {header}")

# Filtrar todas las líneas que NO sean el encabezado
paises_unidades_sin_header = paises_unidades.filter(lambda linea: linea != header)

paises_unidades_sin_header.take(5)
# Salida:
# [('Germany', '1'),
#  ('Germany', '1'),
#  ('Germany', '1'),
#  ...]

Encabezado: ('Country', 'Units')


[('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1')]

In [29]:
# Paso 6 Convertir a Enteros y sumar por pais
# Convertir unidades a enteros
paises_unidades_int = paises_unidades_sin_header.map(
    lambda x: (x[0], int(x[1]))
)

# Sumar unidades por país usando reduceByKey
paises_total_unidades = paises_unidades_int.reduceByKey(lambda a, b: a + b)

# Obtener resultados
resultados = paises_total_unidades.collect()

# Mostrar top 10 países por unidades vendidas
top10 = sorted(resultados, key=lambda x: x[1], reverse=True)[:10]
for pais, unidades in top10:
    print(f"{pais}: {unidades:,} unidades")

Germany: 31,746 unidades
France: 31,739 unidades
Canada: 31,148 unidades
Mexico: 31,095 unidades


In [30]:
# Seccion 8. Cerrar sesión
spark.stop()